In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [8]:
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.


In [9]:
import duckdb

print("DuckDB version:", duckdb.__version__)

con = duckdb.connect(":memory:")

result = con.execute("""
    SELECT
        'DuckDB is ready' AS status,
        1 + 1 AS test_result
""").fetchdf()

display(result)

DuckDB version: 1.3.2


,status,test_result
0,DuckDB is ready,2


In [1]:

import duckdb

con = duckdb.connect(":memory:")

# Aktifkan dukungan akses file melalui HTTP.
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

source_url = (
    "https://huggingface.co/datasets/"
    "openfoodfacts/product-database/resolve/main/food.parquet"
)

# Periksa struktur kolom tanpa meminta seluruh isi tabel.
schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{source_url}')"
).fetchdf()

display(schema.head(20))

print("Jumlah kolom:", len(schema))

,column_name,column_type,null,key,default,extra
0,additives_n,INTEGER,YES,None,None,None
1,additives_tags,VARCHAR[],YES,None,None,None
2,allergens_tags,VARCHAR[],YES,None,None,None
3,brands_tags,VARCHAR[],YES,None,None,None
4,brands,VARCHAR,YES,None,None,None
5,categories,VARCHAR,YES,None,None,None
6,categories_tags,VARCHAR[],YES,None,None,None
7,categories_properties,"STRUCT(ciqual_food_code INTEGER, agribalyse_fo...",YES,None,None,None
8,checkers_tags,VARCHAR[],YES,None,None,None
9,ciqual_food_name_tags,VARCHAR[],YES,None,None,None


Jumlah kolom: 111


In [2]:

source_url = (
    "https://huggingface.co/datasets/"
    "openfoodfacts/product-database/resolve/main/food.parquet"
)

sample = con.execute("""
    SELECT
        code,
        brands,
        brands_tags,
        categories,
        categories_tags,
        countries_tags,
        completeness
    FROM read_parquet(?)
    LIMIT 20
""", [source_url]).fetchdf()

display(sample)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,code,brands,brands_tags,categories,categories_tags,countries_tags,completeness
0,0000101209159,Bovetti,[xx:Bovetti],None,"[en:breakfasts, en:spreads, en:sweet-spreads, ...",[en:france],0.7875
1,0000105000011,Lagg's,[lagg-s],null,[en:null],[en:united-states],0.6000
2,0000105000042,Lagg's,[lagg-s],"Plant-based foods and beverages, Beverages, Ho...","[en:plant-based-foods-and-beverages, en:bevera...",[en:united-states],0.5000
3,0000105000059,Lagg's,[xx:lagg-s],"Beverages and beverages preparations, Plant-ba...","[en:beverages-and-beverages-preparations, en:p...",[en:united-states],0.6000
4,0000105000073,Lagg's,[xx:Lagg's],None,[],[en:united-states],0.3000
5,0000105000196,Lagg's,[lagg-s],null,[en:null],[en:united-states],0.6000
6,0000105000219,Lagg's,[lagg-s],null,[en:null],[en:united-states],0.6000
7,0000105000318,Lagg's,[lagg-s],null,[en:null],[en:united-states],0.6000
8,0000105000356,Lagg's,[lagg-s],"Plant-based foods and beverages, Beverages, Ho...","[en:plant-based-foods-and-beverages, en:bevera...",[en:united-states],0.5000
9,0000105000363,Lagg's,[lagg-s],null,[en:null],[en:united-states],0.6000


In [2]:

import duckdb

print("DuckDB version:", duckdb.__version__)

con = duckdb.connect(":memory:")

print("Koneksi DuckDB berhasil dibuat.")


DuckDB version: 1.3.2
Koneksi DuckDB berhasil dibuat.


In [3]:

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

source_url = (
    "https://huggingface.co/datasets/"
    "openfoodfacts/product-database/resolve/main/food.parquet"
)

print("Akses Parquet siap diuji.")


Akses Parquet siap diuji.


In [4]:

test = con.execute("""
    SELECT code, brands, countries_tags
    FROM read_parquet(?)
    LIMIT 5
""", [source_url]).fetchdf()

display(test)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,code,brands,countries_tags
0,0000101209159,Bovetti,[en:france]
1,0000105000011,Lagg's,[en:united-states]
2,0000105000042,Lagg's,[en:united-states]
3,0000105000059,Lagg's,[en:united-states]
4,0000105000073,Lagg's,[en:united-states]


In [5]:

sample_profile = con.execute("""
    SELECT
        COUNT(*) AS sample_rows,
        COUNT(code) AS non_null_codes,
        COUNT(DISTINCT code) AS distinct_codes,
        COUNT(brands) AS non_null_brands,
        COUNT(categories_tags) AS non_null_category_tags,
        COUNT(countries_tags) AS non_null_country_tags,
        COUNT(completeness) AS non_null_completeness
    FROM (
        SELECT
            code,
            brands,
            categories_tags,
            countries_tags,
            completeness
        FROM read_parquet(?)
        LIMIT 1000
    ) AS sample
""", [source_url]).fetchdf()

display(sample_profile)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,sample_rows,non_null_codes,distinct_codes,non_null_brands,non_null_category_tags,non_null_country_tags,non_null_completeness
0,1000,1000,1000,936,906,1000,1000


In [6]:

country_sample = con.execute("""
    SELECT
        countries_tags,
        COUNT(*) AS product_records
    FROM (
        SELECT countries_tags
        FROM read_parquet(?)
        LIMIT 1000
    ) AS sample
    GROUP BY countries_tags
    ORDER BY product_records DESC
    LIMIT 15
""", [source_url]).fetchdf()

display(country_sample)


,countries_tags,product_records
0,[en:united-states],801
1,[en:france],123
2,"[en:united-states, en:world]",14
3,[en:germany],12
4,"[en:france, en:united-kingdom]",11
5,[en:united-kingdom],9
6,"[en:france, en:germany]",4
7,"[en:france, en:united-states]",4
8,"[en:canada, en:united-states]",1
9,"[en:bolivia, en:united-states]",1


In [7]:

country_quality = con.execute("""
    SELECT
        COUNT(*) AS sample_rows,

        SUM(
            CASE
                WHEN countries_tags = []
                THEN 1 ELSE 0
            END
        ) AS empty_country_arrays,

        SUM(
            CASE
                WHEN list_contains(countries_tags, 'en:null')
                THEN 1 ELSE 0
            END
        ) AS contains_null_tag,

        SUM(
            CASE
                WHEN list_contains(countries_tags, 'en:world')
                THEN 1 ELSE 0
            END
        ) AS contains_world_tag,

        SUM(
            CASE
                WHEN array_length(countries_tags) > 1
                THEN 1 ELSE 0
            END
        ) AS multiple_country_tags

    FROM (
        SELECT countries_tags
        FROM read_parquet(?)
        LIMIT 1000
    ) AS sample
""", [source_url]).fetchdf()

display(country_quality)


,sample_rows,empty_country_arrays,contains_null_tag,contains_world_tag,multiple_country_tags
0,1000,0.0,0.0,16.0,47.0


In [8]:

indonesia_sample = con.execute("""
    SELECT
        COUNT(*) AS tagged_indonesia_records,
        COUNT(DISTINCT code) AS distinct_product_codes
    FROM (
        SELECT
            code,
            countries_tags
        FROM read_parquet(?)
        LIMIT 1000
    ) AS sample
    WHERE list_contains(countries_tags, 'en:indonesia')
""", [source_url]).fetchdf()

display(indonesia_sample)


,tagged_indonesia_records,distinct_product_codes
0,0,0


In [9]:

parquet_info = con.execute("""
    SELECT *
    FROM parquet_file_metadata(?)
""", [source_url]).fetchdf()

display(parquet_info)


,file_name,created_by,num_rows,num_row_groups,format_version,encryption_algorithm,footer_signing_key_metadata
0,https://huggingface.co/datasets/openfoodfacts/...,parquet-cpp-arrow version 18.0.0-SNAPSHOT,4828207,4716,2,None,None


In [10]:

row_group_stats = con.execute("""
    SELECT *
    FROM parquet_metadata(?)
    WHERE path_in_schema = 'countries_tags'
    LIMIT 10
""", [source_url]).fetchdf()

display(row_group_stats)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,file_name,row_group_id,row_group_num_rows,row_group_num_columns,row_group_bytes,column_id,file_offset,num_values,path_in_schema,type,...,index_page_offset,dictionary_page_offset,data_page_offset,total_compressed_size,total_uncompressed_size,key_value_metadata,bloom_filter_offset,bloom_filter_length,min_is_exact,max_is_exact


In [11]:

metadata_columns = con.execute("""
    SELECT DISTINCT path_in_schema
    FROM parquet_metadata(?)
    ORDER BY path_in_schema
    LIMIT 30
""", [source_url]).fetchdf()

display(metadata_columns)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,path_in_schema
0,additives_n
1,"additives_tags, list, element"
2,"allergens_tags, list, element"
3,brands
4,"brands_tags, list, element"
5,categories
6,"categories_properties, agribalyse_food_code"
7,"categories_properties, agribalyse_proxy_food_code"
8,"categories_properties, ciqual_food_code"
9,"categories_tags, list, element"


In [1]:

import requests

api_url = "https://world.openfoodfacts.org/api/v2/search"

params = {
    "countries_tags_en": "indonesia",
    "page_size": 1,
    "fields": "code,brands,categories_tags,countries_tags,completeness"
}

try:
    response = requests.get(
        api_url,
        params=params,
        headers={"User-Agent": "OFF-Brand-Landscape-Portfolio/1.0"},
        timeout=20
    )

    print("HTTP status:", response.status_code)
    response.raise_for_status()

    result = response.json()

    print("Total matching records reported by API:", result.get("count"))
    print("Records returned:", len(result.get("products", [])))

    if result.get("products"):
        display(result["products"])

except requests.exceptions.Timeout:
    print("Request timed out after 20 seconds.")

except requests.exceptions.RequestException as error:
    print("Request failed:", error)


HTTP status: 503
Request failed: 503 Server Error: Service Temporarily Unavailable for url: https://world.openfoodfacts.org/api/v2/search?countries_tags_en=indonesia&page_size=1&fields=code%2Cbrands%2Ccategories_tags%2Ccountries_tags%2Ccompleteness


In [2]:

import shutil

disk = shutil.disk_usage("/kaggle/working")

print(f"Total disk: {disk.total / (1024**3):.2f} GB")
print(f"Used disk: {disk.used / (1024**3):.2f} GB")
print(f"Free disk: {disk.free / (1024**3):.2f} GB")


Total disk: 19.52 GB
Used disk: 0.00 GB
Free disk: 19.50 GB


In [3]:

import psutil

memory = psutil.virtual_memory()

print(f"Total RAM: {memory.total / (1024**3):.2f} GB")
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")


Total RAM: 31.35 GB
Available RAM: 30.01 GB


In [4]:

import time

start = time.time()

try:
    test = con.execute("""
        SELECT code, brands, countries_tags
        FROM read_parquet(?)
        LIMIT 1000
    """, [source_url]).fetchdf()

    print("Rows returned:", len(test))
    print(f"Elapsed seconds: {time.time() - start:.1f}")

except Exception as e:
    print(type(e).__name__, str(e))


NameError name 'con' is not defined


In [5]:

import duckdb
import time

con = duckdb.connect(":memory:")

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

source_url = (
    "https://huggingface.co/datasets/"
    "openfoodfacts/product-database/resolve/main/food.parquet"
)

print("DuckDB:", duckdb.__version__)
print("Connection ready:", con is not None)
print("Source URL ready:", source_url)


DuckDB: 1.3.2
Connection ready: True
Source URL ready: https://huggingface.co/datasets/openfoodfacts/product-database/resolve/main/food.parquet


In [6]:

start = time.time()

metadata = con.execute("""
    SELECT
        COUNT(*) AS metadata_entries,
        COUNT(DISTINCT row_group_id) AS row_groups
    FROM parquet_metadata(?)
""", [source_url]).fetchdf()

display(metadata)
print(f"Elapsed seconds: {time.time() - start:.1f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,metadata_entries,row_groups
0,683820,4716


Elapsed seconds: 8.8


In [7]:

import requests

response = requests.head(
    source_url,
    headers={"User-Agent": "OFF-Brand-Landscape-Portfolio/1.0"},
    allow_redirects=True,
    timeout=20
)

print("HTTP status:", response.status_code)

size = response.headers.get("Content-Length")
if size is not None:
    print(f"File size: {int(size) / (1024**3):.2f} GiB")
else:
    print("File size: not provided by server")

print("Accept-Ranges:", response.headers.get("Accept-Ranges"))


HTTP status: 200
File size: 7.43 GiB
Accept-Ranges: bytes


In [8]:

import os
import requests
from IPython.display import display, clear_output

local_path = "/kaggle/working/food.parquet"
expected_size = int(response.headers["Content-Length"])
chunk_size = 8 * 1024 * 1024

downloaded = 0

with requests.get(
    source_url,
    headers={"User-Agent": "OFF-Brand-Landscape-Portfolio/1.0"},
    stream=True,
    timeout=(20, 120)
) as r:
    r.raise_for_status()

    with open(local_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)

                if downloaded % (256 * 1024 * 1024) < chunk_size:
                    print(
                        f"Downloaded: {downloaded / (1024**3):.2f} / "
                        f"{expected_size / (1024**3):.2f} GiB"
                    )

actual_size = os.path.getsize(local_path)

print("\nDownload finished.")
print(f"Expected size: {expected_size:,} bytes")
print(f"Actual size:   {actual_size:,} bytes")
print(f"Size matches:  {actual_size == expected_size}")


Downloaded: 0.25 / 7.43 GiB
Downloaded: 0.50 / 7.43 GiB
Downloaded: 0.75 / 7.43 GiB
Downloaded: 1.00 / 7.43 GiB
Downloaded: 1.25 / 7.43 GiB
Downloaded: 1.50 / 7.43 GiB
Downloaded: 1.75 / 7.43 GiB
Downloaded: 2.00 / 7.43 GiB
Downloaded: 2.25 / 7.43 GiB
Downloaded: 2.50 / 7.43 GiB
Downloaded: 2.75 / 7.43 GiB
Downloaded: 3.00 / 7.43 GiB
Downloaded: 3.25 / 7.43 GiB
Downloaded: 3.50 / 7.43 GiB
Downloaded: 3.75 / 7.43 GiB
Downloaded: 4.00 / 7.43 GiB
Downloaded: 4.25 / 7.43 GiB
Downloaded: 4.50 / 7.43 GiB
Downloaded: 4.75 / 7.43 GiB
Downloaded: 5.00 / 7.43 GiB
Downloaded: 5.25 / 7.43 GiB
Downloaded: 5.50 / 7.43 GiB
Downloaded: 5.75 / 7.43 GiB
Downloaded: 6.00 / 7.43 GiB
Downloaded: 6.25 / 7.43 GiB
Downloaded: 6.50 / 7.43 GiB
Downloaded: 6.75 / 7.43 GiB
Downloaded: 7.00 / 7.43 GiB
Downloaded: 7.25 / 7.43 GiB

Download finished.
Expected size: 7,975,461,945 bytes
Actual size:   7,975,461,945 bytes
Size matches:  True


In [9]:

import os
import duckdb
import pandas as pd

local_path = "/kaggle/working/food.parquet"

assert os.path.exists(local_path), "File tidak ditemukan."
assert os.path.getsize(local_path) == 7_975_461_945, "Ukuran file tidak sesuai."

con = duckdb.connect(":memory:")

# 1. Periksa metadata file Parquet
file_meta = con.execute(
    "SELECT * FROM parquet_file_metadata(?)",
    [local_path]
).fetchdf()

# 2. Periksa struktur kolom
schema = con.execute(
    "DESCRIBE SELECT * FROM read_parquet(?)",
    [local_path]
).fetchdf()

# 3. Hitung jumlah baris yang dapat dibaca
row_count = con.execute(
    "SELECT COUNT(*) AS total_rows FROM read_parquet(?)",
    [local_path]
).fetchone()[0]

print("File size:", f"{os.path.getsize(local_path):,}", "bytes")
print("Total rows:", f"{row_count:,}")
print("Total columns:", len(schema))
print("\nSchema preview:")
display(schema.head(20))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

File size: 7,975,461,945 bytes
Total rows: 4,828,207
Total columns: 111

Schema preview:


,column_name,column_type,null,key,default,extra
0,additives_n,INTEGER,YES,None,None,None
1,additives_tags,VARCHAR[],YES,None,None,None
2,allergens_tags,VARCHAR[],YES,None,None,None
3,brands_tags,VARCHAR[],YES,None,None,None
4,brands,VARCHAR,YES,None,None,None
5,categories,VARCHAR,YES,None,None,None
6,categories_tags,VARCHAR[],YES,None,None,None
7,categories_properties,"STRUCT(ciqual_food_code INTEGER, agribalyse_fo...",YES,None,None,None
8,checkers_tags,VARCHAR[],YES,None,None,None
9,ciqual_food_name_tags,VARCHAR[],YES,None,None,None


In [10]:

import pandas as pd
from IPython.display import display

profile = con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(code) AS non_null_codes,
        COUNT(DISTINCT code) AS distinct_non_null_codes,
        COUNT(*) FILTER (
            WHERE code IS NULL OR TRIM(code) = ''
        ) AS missing_or_empty_codes,

        COUNT(*) FILTER (
            WHERE brands IS NULL OR TRIM(brands) = ''
        ) AS missing_or_empty_brands,

        COUNT(*) FILTER (
            WHERE brands_tags IS NULL
        ) AS null_brand_tag_arrays,
        COUNT(*) FILTER (
            WHERE brands_tags IS NOT NULL
              AND len(brands_tags) = 0
        ) AS empty_brand_tag_arrays,

        COUNT(*) FILTER (
            WHERE categories_tags IS NULL
        ) AS null_category_tag_arrays,
        COUNT(*) FILTER (
            WHERE categories_tags IS NOT NULL
              AND len(categories_tags) = 0
        ) AS empty_category_tag_arrays,

        COUNT(*) FILTER (
            WHERE countries_tags IS NULL
        ) AS null_country_tag_arrays,
        COUNT(*) FILTER (
            WHERE countries_tags IS NOT NULL
              AND len(countries_tags) = 0
        ) AS empty_country_tag_arrays,

        COUNT(*) FILTER (
            WHERE list_contains(countries_tags, 'en:indonesia')
        ) AS rows_tagged_indonesia,
        COUNT(*) FILTER (
            WHERE list_contains(countries_tags, 'en:world')
        ) AS rows_tagged_world,
        COUNT(*) FILTER (
            WHERE len(countries_tags) > 1
        ) AS rows_with_multiple_country_tags,

        COUNT(*) FILTER (
            WHERE completeness IS NULL
        ) AS null_completeness,
        MIN(completeness) AS min_completeness,
        MAX(completeness) AS max_completeness,
        COUNT(*) FILTER (
            WHERE completeness < 0 OR completeness > 1
        ) AS completeness_outside_0_1

    FROM read_parquet(?)
""", [local_path]).fetchdf()

display(
    profile.T.rename(columns={0: "value"})
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,value
total_rows,4828207.0
non_null_codes,4828207.0
distinct_non_null_codes,4828146.0
missing_or_empty_codes,1.0
missing_or_empty_brands,1733777.0
null_brand_tag_arrays,1620744.0
empty_brand_tag_arrays,113245.0
null_category_tag_arrays,2732642.0
empty_category_tag_arrays,159402.0
null_country_tag_arrays,9797.0


In [11]:

duplicate_audit = con.execute("""
    WITH code_counts AS (
        SELECT
            code,
            COUNT(*) AS row_count
        FROM read_parquet(?)
        WHERE code IS NOT NULL
          AND TRIM(code) <> ''
        GROUP BY code
    )
    SELECT
        COUNT(*) AS distinct_valid_codes,
        COUNT(*) FILTER (
            WHERE row_count > 1
        ) AS repeated_codes,
        SUM(
            CASE WHEN row_count > 1
                 THEN row_count ELSE 0 END
        ) AS rows_in_repeated_groups,
        SUM(
            CASE WHEN row_count > 1
                 THEN row_count - 1 ELSE 0 END
        ) AS excess_rows_if_one_per_code,
        MAX(row_count) AS max_rows_per_code
    FROM code_counts
""", [local_path]).fetchdf()

display(duplicate_audit.T.rename(columns={0: "value"}))


,value
distinct_valid_codes,4828145.0
repeated_codes,61.0
rows_in_repeated_groups,122.0
excess_rows_if_one_per_code,61.0
max_rows_per_code,2.0


In [12]:

duplicate_examples = con.execute("""
    WITH repeated_codes AS (
        SELECT code
        FROM read_parquet(?)
        WHERE code IS NOT NULL
          AND TRIM(code) <> ''
        GROUP BY code
        HAVING COUNT(*) > 1
    )
    SELECT
        p.code,
        p.brands,
        p.brands_tags,
        p.categories_tags,
        p.countries_tags,
        p.created_t,
        p.completeness
    FROM read_parquet(?) AS p
    INNER JOIN repeated_codes AS d
        ON p.code = d.code
    ORDER BY p.code
    LIMIT 20
""", [local_path, local_path]).fetchdf()

display(duplicate_examples)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,code,brands,brands_tags,categories_tags,countries_tags,created_t,completeness
0,0059527070430,Taste Of Nature,[taste-of-nature],"[en:snacks, en:sweet-snacks, en:bars, en:cerea...","[en:canada, en:france, en:world]",1501498939,0.8500
1,0059527070430,Taste of Nature,[taste-of-nature],"[en:snacks, en:sweet-snacks, en:bars, fr:barre...",[en:france],1732542328,0.7000
2,0059527171687,Taste of Nature,[taste-of-nature],"[en:snacks, en:sweet-snacks, en:bars, fr:barre...",[en:france],1732542881,0.7000
3,0059527171687,Taste of nature,[taste-of-nature],[],[en:france],1505392927,0.5875
4,0059527401555,Taste Of Nature,[taste-of-nature],"[en:snacks, en:sweet-snacks, en:bars, fr:snack...","[en:france, en:world, en:belgique, en:suisse]",1479904278,0.6500
5,0059527401555,Taste of Nature,[taste-of-nature],"[en:snacks, en:sweet-snacks, en:bars, fr:barre...",[en:france],1732540798,0.6875
6,0059527501552,Taste Of Nature,[taste-of-nature],"[en:plant-based-foods-and-beverages, en:plant-...","[en:france, en:world, en:suisse]",1504534990,0.7500
7,0059527501552,Taste of Nature,[taste-of-nature],"[en:snacks, en:sweet-snacks, en:bars, fr:barre...",[en:france],1732539534,0.6875
8,0059527601559,Taste of Nature,[taste-of-nature],"[en:snacks, en:sweet-snacks, en:bars, fr:barre...",[en:france],1732541503,0.7000
9,0059527601559,Taste Of Nature,[taste-of-nature],"[en:plant-based-foods-and-beverages, en:plant-...","[en:france, en:switzerland, en:world]",1479729628,0.7625


In [13]:

exact_duplicate_audit = con.execute("""
    WITH repeated_codes AS (
        SELECT code
        FROM read_parquet(?)
        WHERE code IS NOT NULL
          AND TRIM(code) <> ''
        GROUP BY code
        HAVING COUNT(*) > 1
    ),
    duplicate_rows AS (
        SELECT p.*
        FROM read_parquet(?) AS p
        INNER JOIN repeated_codes AS d
            ON p.code = d.code
    ),
    distinct_rows AS (
        SELECT DISTINCT *
        FROM duplicate_rows
    )
    SELECT
        (SELECT COUNT(*) FROM duplicate_rows)
            AS candidate_rows,
        (SELECT COUNT(*) FROM distinct_rows)
            AS distinct_full_records,
        (SELECT COUNT(*) FROM duplicate_rows)
        - (SELECT COUNT(*) FROM distinct_rows)
            AS exact_duplicate_excess_rows
""", [local_path, local_path]).fetchdf()

display(exact_duplicate_audit)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,candidate_rows,distinct_full_records,exact_duplicate_excess_rows
0,122,122,0


In [16]:

country_profile = con.execute("""
    SELECT
        country_tag,
        COUNT(*) AS product_records
    FROM read_parquet(?)
    CROSS JOIN UNNEST(countries_tags) AS t(country_tag)
    WHERE country_tag IS NOT NULL
      AND country_tag <> 'en:world'
      AND country_tag <> 'en:null'
    GROUP BY country_tag
    ORDER BY product_records DESC
""", [local_path]).fetchdf()

display(country_profile)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,country_tag,product_records
0,en:france,1275718
1,en:united-states,988834
2,en:germany,432272
3,en:spain,360152
4,en:italy,271468
...,...,...
428,en:Middle East,1
429,en:irland,1
430,en:alderney,1
431,en:Sark,1


In [17]:

country_checks = con.execute("""
    SELECT
        country_tag,
        COUNT(*) AS product_records
    FROM read_parquet(?)
    CROSS JOIN UNNEST(countries_tags) AS t(country_tag)
    WHERE country_tag IS NOT NULL
      AND (
          lower(country_tag) LIKE '%indonesia%'
          OR lower(country_tag) LIKE '%irland%'
          OR lower(country_tag) LIKE '%middle east%'
          OR lower(country_tag) LIKE '%piemonte%'
          OR lower(country_tag) LIKE '%sark%'
          OR lower(country_tag) LIKE '%alderney%'
          OR lower(country_tag) LIKE '%world%'
          OR lower(country_tag) LIKE '%null%'
      )
    GROUP BY country_tag
    ORDER BY product_records DESC, country_tag
""", [local_path]).fetchdf()

display(country_checks)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,country_tag,product_records
0,en:world,72940
1,en:indonesia,9118
2,en:worldwide,35
3,en:Middle East,1
4,en:Middle east,1
5,en:Piemonte,1
6,en:Sark,1
7,en:WORLDWIDE,1
8,en:alderney,1
9,en:irland,1


In [18]:

country_candidates = con.execute("""
    SELECT
        country_tag,
        COUNT(*) AS product_records
    FROM read_parquet(?)
    CROSS JOIN UNNEST(countries_tags) AS t(country_tag)
    WHERE country_tag IS NOT NULL
      AND lower(country_tag) NOT IN (
          'en:world',
          'en:worldwide',
          'en:middle east',
          'en:piemonte',
          'en:sark',
          'en:alderney'
      )
    GROUP BY country_tag
    ORDER BY product_records DESC
""", [local_path]).fetchdf()

print("Jumlah label setelah pengecualian awal:",
      len(country_candidates))
display(country_candidates)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Jumlah label setelah pengecualian awal: 426


,country_tag,product_records
0,en:france,1275718
1,en:united-states,988834
2,en:germany,432272
3,en:spain,360152
4,en:italy,271468
...,...,...
421,bg:литвс,1
422,el:ελλαδα,1
423,en:Stany Zjednoczone,1
424,en:comidosil,1


In [19]:

language_prefix_audit = con.execute("""
    WITH country_tags AS (
        SELECT
            country_tag,
            regexp_extract(
                country_tag, '^([^:]+):', 1
            ) AS language_prefix,
            COUNT(*) AS product_records
        FROM read_parquet(?)
        CROSS JOIN UNNEST(countries_tags) AS t(country_tag)
        WHERE country_tag IS NOT NULL
        GROUP BY country_tag
    )
    SELECT
        CASE
            WHEN language_prefix = ''
                THEN '(no prefix)'
            ELSE lower(language_prefix)
        END AS language_prefix,
        COUNT(*) AS distinct_labels,
        SUM(product_records) AS tag_assignments
    FROM country_tags
    GROUP BY 1
    ORDER BY distinct_labels DESC
""", [local_path]).fetchdf()

display(language_prefix_audit)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,language_prefix,distinct_labels,tag_assignments
0,en,365,5079446.0
1,fr,23,305.0
2,es,15,37.0
3,de,9,11.0
4,ca,4,5.0
5,nl,3,33.0
6,zh,3,8.0
7,ar,2,2.0
8,it,2,2.0
9,ru,2,31.0


In [11]:
import os

print("File di /kaggle/working:")
print(os.listdir("/kaggle/working"))

print("\nDirektori input Kaggle:")
print(os.listdir("/kaggle/input"))

File di /kaggle/working:
['.virtual_documents']

Direktori input Kaggle:
[]


In [8]:
import duckdb

con = duckdb.connect()

local_path = (
    "/kaggle/input/datasets/maulanakinditamami/"
    "open-food-facts-product-database-salin-judul/food.parquet"
)

result = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE code IS NOT NULL) AS non_null_codes
    FROM read_parquet(?)
""", [local_path]).fetchdf()

display(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,non_null_codes
0,4828207,4828207


In [9]:

country_shortlist = con.execute("""
    SELECT
        country_tag,
        COUNT(*) AS product_records
    FROM read_parquet(?)
    CROSS JOIN UNNEST(countries_tags) AS t(country_tag)
    WHERE country_tag IS NOT NULL
      AND regexp_matches(country_tag, '^en:')
      AND lower(country_tag) NOT IN (
          'en:world',
          'en:worldwide',
          'en:middle east',
          'en:piemonte',
          'en:sark',
          'en:alderney',
          'en:null'
      )
    GROUP BY country_tag
    HAVING COUNT(*) >= 1000
    ORDER BY product_records DESC
""", [local_path]).fetchdf()

print("Negara/label dengan >= 1.000 penetapan tag:",
      len(country_shortlist))
display(country_shortlist)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Negara/label dengan >= 1.000 penetapan tag: 107


,country_tag,product_records
0,en:france,1275718
1,en:united-states,988834
2,en:germany,432272
3,en:spain,360152
4,en:italy,271468
...,...,...
102,en:virgin-islands-of-the-united-states,1172
103,en:cote-d-ivoire,1146
104,en:albania,1135
105,en:trinidad-and-tobago,1113


In [3]:
import duckdb
import os

local_path = (
    "/kaggle/input/datasets/maulanakinditamami/"
    "open-food-facts-product-database-salin-judul/food.parquet"
)

assert os.path.exists(local_path), "File dataset tidak ditemukan!"

con = duckdb.connect()

print("DuckDB siap.")
print("Dataset ditemukan:", os.path.exists(local_path))

DuckDB siap.
Dataset ditemukan: True


In [4]:
country_shortlist = con.execute("""
    SELECT
        country_tag,
        COUNT(*) AS product_records
    FROM read_parquet(?)
    CROSS JOIN UNNEST(countries_tags) AS t(country_tag)
    WHERE country_tag IS NOT NULL
      AND regexp_matches(country_tag, '^en:')
      AND lower(country_tag) NOT IN (
          'en:world',
          'en:worldwide',
          'en:middle east',
          'en:piemonte',
          'en:sark',
          'en:alderney',
          'en:null'
      )
    GROUP BY country_tag
    HAVING COUNT(*) >= 1000
    ORDER BY product_records DESC
""", [local_path]).fetchdf()

print("Negara/label dengan >= 1.000 penetapan tag:",
      len(country_shortlist))

display(
    country_shortlist[
        country_shortlist["country_tag"].str.contains(
            "indonesia|malaysia|singapore|thailand|vietnam|philippines",
            case=False,
            na=False
        )
    ]
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Negara/label dengan >= 1.000 penetapan tag: 107


,country_tag,product_records
32,en:singapore,11986
36,en:thailand,11314
42,en:philippines,9176
43,en:indonesia,9118
47,en:malaysia,7426
85,en:vietnam,1653


In [5]:
duplicate_audit = con.execute("""
    WITH code_counts AS (
        SELECT
            code,
            COUNT(*) AS row_count
        FROM read_parquet(?)
        WHERE code IS NOT NULL
          AND TRIM(code) <> ''
        GROUP BY code
    )
    SELECT
        COUNT(*) AS distinct_valid_codes,
        COUNT(*) FILTER (
            WHERE row_count > 1
        ) AS repeated_codes,
        SUM(
            CASE
                WHEN row_count > 1 THEN row_count
                ELSE 0
            END
        ) AS rows_in_repeated_groups,
        SUM(
            CASE
                WHEN row_count > 1 THEN row_count - 1
                ELSE 0
            END
        ) AS excess_rows_if_one_per_code
    FROM code_counts
""", [local_path]).fetchdf()

display(duplicate_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,distinct_valid_codes,repeated_codes,rows_in_repeated_groups,excess_rows_if_one_per_code
0,4828145,61,122.0,61.0


In [6]:
con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(?)
""", [local_path]).fetchdf()

,column_name,column_type,null,key,default,extra
0,additives_n,INTEGER,YES,None,None,None
1,additives_tags,VARCHAR[],YES,None,None,None
2,allergens_tags,VARCHAR[],YES,None,None,None
3,brands_tags,VARCHAR[],YES,None,None,None
4,brands,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
106,unknown_nutrients_tags,VARCHAR[],YES,None,None,None
107,vitamins_tags,VARCHAR[],YES,None,None,None
108,with_non_nutritive_sweeteners,INTEGER,YES,None,None,None
109,with_sweeteners,INTEGER,YES,None,None,None


In [7]:
relevant_columns = con.execute("""
    DESCRIBE
    SELECT
        code,
        brands,
        brands_tags,
        categories,
        categories_tags,
        countries_tags,
        completeness,
        created_t
    FROM read_parquet(?)
""", [local_path]).fetchdf()

display(relevant_columns)

,column_name,column_type,null,key,default,extra
0,code,VARCHAR,YES,None,None,None
1,brands,VARCHAR,YES,None,None,None
2,brands_tags,VARCHAR[],YES,None,None,None
3,categories,VARCHAR,YES,None,None,None
4,categories_tags,VARCHAR[],YES,None,None,None
5,countries_tags,VARCHAR[],YES,None,None,None
6,completeness,FLOAT,YES,None,None,None
7,created_t,BIGINT,YES,None,None,None


In [8]:
array_profile = con.execute("""
    SELECT
        COUNT(*) AS total_records,

        COUNT(*) FILTER (
            WHERE brands_tags IS NULL
        ) AS null_brand_arrays,

        COUNT(*) FILTER (
            WHERE brands_tags IS NOT NULL
              AND len(brands_tags) = 0
        ) AS empty_brand_arrays,

        COUNT(*) FILTER (
            WHERE brands_tags IS NOT NULL
              AND len(brands_tags) > 1
        ) AS multiple_brand_arrays,

        COUNT(*) FILTER (
            WHERE categories_tags IS NOT NULL
              AND len(categories_tags) > 1
        ) AS multiple_category_arrays,

        COUNT(*) FILTER (
            WHERE countries_tags IS NOT NULL
              AND len(countries_tags) > 1
        ) AS multiple_country_arrays
    FROM read_parquet(?)
""", [local_path]).fetchdf()

display(array_profile)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_records,null_brand_arrays,empty_brand_arrays,multiple_brand_arrays,multiple_category_arrays,multiple_country_arrays
0,4828207,1620744,113245,195096,1704272,226015


In [9]:
identity_check = con.execute("""
    SELECT
        COUNT(*) AS total_records,

        COUNT(DISTINCT code) AS distinct_codes,

        COUNT(DISTINCT code) FILTER (
            WHERE code IS NOT NULL
              AND TRIM(code) <> ''
        ) AS distinct_valid_codes,

        COUNT(*) FILTER (
            WHERE code IS NULL
               OR TRIM(code) = ''
        ) AS missing_or_empty_codes,

        COUNT(DISTINCT created_t) AS distinct_created_t,

        COUNT(DISTINCT creator) AS distinct_creators
    FROM read_parquet(?)
""", [local_path]).fetchdf()

display(identity_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_records,distinct_codes,distinct_valid_codes,missing_or_empty_codes,distinct_created_t,distinct_creators
0,4828207,4828146,4828145,1,4488380,46667


In [10]:
# Membuat tabel inti dari snapshot Parquet
con.execute("""
    CREATE OR REPLACE TABLE product_records AS
    SELECT
        ROW_NUMBER() OVER () AS record_id,
        code,
        brands,
        brands_tags,
        categories,
        categories_tags,
        countries_tags,
        completeness,
        created_t
    FROM read_parquet(?)
""", [local_path])

print(
    con.execute("""
        SELECT
            COUNT(*) AS total_records,
            COUNT(DISTINCT record_id) AS unique_record_ids,
            MIN(record_id) AS min_record_id,
            MAX(record_id) AS max_record_id
        FROM product_records
    """).fetchdf()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_records  unique_record_ids  min_record_id  max_record_id
0        4828207            4828207              1        4828207


In [11]:
display(
    con.execute("""
        DESCRIBE product_records
    """).fetchdf()
)

display(
    con.execute("""
        SELECT *
        FROM product_records
        ORDER BY record_id
        LIMIT 10
    """).fetchdf()
)

,column_name,column_type,null,key,default,extra
0,record_id,BIGINT,YES,None,None,None
1,code,VARCHAR,YES,None,None,None
2,brands,VARCHAR,YES,None,None,None
3,brands_tags,VARCHAR[],YES,None,None,None
4,categories,VARCHAR,YES,None,None,None
5,categories_tags,VARCHAR[],YES,None,None,None
6,countries_tags,VARCHAR[],YES,None,None,None
7,completeness,FLOAT,YES,None,None,None
8,created_t,BIGINT,YES,None,None,None


,record_id,code,brands,brands_tags,categories,categories_tags,countries_tags,completeness,created_t
0,1,0000101209159,Bovetti,[xx:Bovetti],None,"[en:breakfasts, en:spreads, en:sweet-spreads, ...",[en:france],0.7875,1519297017
1,2,0000105000011,Lagg's,[lagg-s],null,[en:null],[en:united-states],0.6000,1489055330
2,3,0000105000042,Lagg's,[lagg-s],"Plant-based foods and beverages, Beverages, Ho...","[en:plant-based-foods-and-beverages, en:bevera...",[en:united-states],0.5000,1489055331
3,4,0000105000059,Lagg's,[xx:lagg-s],"Beverages and beverages preparations, Plant-ba...","[en:beverages-and-beverages-preparations, en:p...",[en:united-states],0.6000,1489055331
4,5,0000105000073,Lagg's,[xx:Lagg's],None,[],[en:united-states],0.3000,1489096571
5,6,0000105000196,Lagg's,[lagg-s],null,[en:null],[en:united-states],0.6000,1489096571
6,7,0000105000219,Lagg's,[lagg-s],null,[en:null],[en:united-states],0.6000,1489055330
7,8,0000105000318,Lagg's,[lagg-s],null,[en:null],[en:united-states],0.6000,1489055331
8,9,0000105000356,Lagg's,[lagg-s],"Plant-based foods and beverages, Beverages, Ho...","[en:plant-based-foods-and-beverages, en:bevera...",[en:united-states],0.5000,1489055331
9,10,0000105000363,Lagg's,[lagg-s],null,[en:null],[en:united-states],0.6000,1489055331


In [12]:
import os

sample_path = "/kaggle/working/product_records_sample.csv"

con.execute("""
    COPY (
        SELECT *
        FROM product_records
        ORDER BY record_id
        LIMIT 100
    )
    TO '/kaggle/working/product_records_sample.csv'
    (HEADER, DELIMITER ',')
""")

print("File exists:", os.path.exists(sample_path))
print("File size:", round(os.path.getsize(sample_path) / 1024, 2), "KB")
print("File path:", sample_path)

File exists: True
File size: 20.14 KB
File path: /kaggle/working/product_records_sample.csv


In [13]:
con.execute("""
    CREATE OR REPLACE TABLE product_countries AS
    SELECT DISTINCT
        p.record_id,
        country_tag
    FROM product_records AS p
    CROSS JOIN UNNEST(p.countries_tags) AS t(country_tag)
    WHERE country_tag IS NOT NULL
      AND TRIM(country_tag) <> ''
""")

display(
    con.execute("""
        SELECT *
        FROM product_countries
        ORDER BY record_id, country_tag
        LIMIT 10
    """).fetchdf()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,record_id,country_tag
0,1,en:france
1,2,en:united-states
2,3,en:united-states
3,4,en:united-states
4,5,en:united-states
5,6,en:united-states
6,7,en:united-states
7,8,en:united-states
8,9,en:united-states
9,10,en:united-states


In [14]:
display(
    con.execute("""
        SELECT
            COUNT(*) AS total_record_country_pairs,
            COUNT(DISTINCT record_id) AS records_with_country_tags,
            COUNT(DISTINCT country_tag) AS distinct_country_tags,
            COUNT(*) - COUNT(DISTINCT record_id) AS additional_country_assignments
        FROM product_countries
    """).fetchdf()
)

,total_record_country_pairs,records_with_country_tags,distinct_country_tags,additional_country_assignments
0,5079894,4805349,434,274545


In [15]:
con.execute("""
    CREATE OR REPLACE TABLE product_categories AS
    SELECT DISTINCT
        p.record_id,
        category_tag
    FROM product_records AS p
    CROSS JOIN UNNEST(p.categories_tags) AS t(category_tag)
    WHERE category_tag IS NOT NULL
      AND TRIM(category_tag) <> ''
""")

display(
    con.execute("""
        SELECT *
        FROM product_categories
        ORDER BY record_id, category_tag
        LIMIT 10
    """).fetchdf()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,record_id,category_tag
0,1,en:breakfasts
1,1,en:chocolate-spreads
2,1,en:cocoa-and-hazelnuts-spreads
3,1,en:hazelnut-spreads
4,1,en:spreads
5,1,en:sweet-spreads
6,1,fr:pates-a-tartiner
7,2,en:null
8,3,en:beverages
9,3,en:hot-beverages


In [16]:
display(
    con.execute("""
        SELECT
            COUNT(*) AS total_record_category_pairs,
            COUNT(DISTINCT record_id) AS records_with_category_tags,
            COUNT(DISTINCT category_tag) AS distinct_category_tags,
            COUNT(*) - COUNT(DISTINCT record_id)
                AS additional_category_assignments
        FROM product_categories
    """).fetchdf()
)

,total_record_category_pairs,records_with_category_tags,distinct_category_tags,additional_category_assignments
0,10480049,1936163,116105,8543886


In [17]:
display(
    con.execute("""
        SELECT
            category_tag,
            COUNT(*) AS record_count
        FROM product_categories
        WHERE LOWER(TRIM(category_tag)) IN (
            'en:null',
            'en:unknown',
            'en:undefined'
        )
        GROUP BY category_tag
        ORDER BY record_count DESC
    """).fetchdf()
)

,category_tag,record_count
0,en:undefined,34795
1,en:null,1192
2,en:Undefined,179
3,en:unknown,57
4,en:Null,5
5,en:Unknown,1


In [18]:
con.execute("""
    CREATE OR REPLACE TABLE product_brands AS
    SELECT DISTINCT
        p.record_id,
        brand_tag
    FROM product_records AS p
    CROSS JOIN UNNEST(p.brands_tags) AS t(brand_tag)
    WHERE brand_tag IS NOT NULL
      AND TRIM(brand_tag) <> ''
""")

display(
    con.execute("""
        SELECT *
        FROM product_brands
        ORDER BY record_id, brand_tag
        LIMIT 10
    """).fetchdf()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,record_id,brand_tag
0,1,xx:Bovetti
1,2,lagg-s
2,3,lagg-s
3,4,xx:lagg-s
4,5,xx:Lagg's
5,6,lagg-s
6,7,lagg-s
7,8,lagg-s
8,9,lagg-s
9,10,lagg-s


In [19]:
display(
    con.execute("""
        SELECT
            COUNT(*) AS total_record_brand_pairs,
            COUNT(DISTINCT record_id) AS records_with_brand_tags,
            COUNT(DISTINCT brand_tag) AS distinct_brand_tags,
            COUNT(*) - COUNT(DISTINCT record_id)
                AS additional_brand_assignments
        FROM product_brands
    """).fetchdf()
)

,total_record_brand_pairs,records_with_brand_tags,distinct_brand_tags,additional_brand_assignments
0,3313249,3094218,539516,219031


In [20]:
display(
    con.execute("""
        SELECT
            p.brands,
            b.brand_tag,
            COUNT(*) AS record_count
        FROM product_brands AS b
        JOIN product_records AS p
            ON b.record_id = p.record_id
        WHERE p.brands IS NOT NULL
          AND TRIM(p.brands) <> ''
        GROUP BY
            p.brands,
            b.brand_tag
        ORDER BY record_count DESC
        LIMIT 30
    """).fetchdf()
)

,brands,brand_tag,record_count
0,Coop,xx:coop,14413
1,BonÀrea,xx:bonarea,12033
2,Aldi,xx:aldi,11274
3,Auchan,xx:auchan,10546
4,U,xx:U,10487
5,Lidl,xx:lidl,10399
6,Carrefour,xx:Carrefour,10267
7,Carrefour,xx:carrefour,9943
8,Hacendado,xx:hacendado,9862
9,Nestlé,xx:nestle,9046


In [21]:
display(
    con.execute("""
        WITH normalized AS (
            SELECT
                brand_tag,
                LOWER(
                    REGEXP_REPLACE(
                        TRIM(brand_tag),
                        '^xx:',
                        ''
                    )
                ) AS normalized_tag
            FROM product_brands
        ),
        tag_groups AS (
            SELECT
                normalized_tag,
                COUNT(DISTINCT brand_tag) AS original_tag_variants
            FROM normalized
            GROUP BY normalized_tag
        )
        SELECT
            COUNT(*) AS normalized_tag_groups,
            SUM(original_tag_variants) AS original_distinct_tags,
            COUNT(*) FILTER (
                WHERE original_tag_variants > 1
            ) AS groups_with_multiple_variants,
            SUM(original_tag_variants - 1) AS excess_tag_variants
        FROM tag_groups
    """).fetchdf()
)

,normalized_tag_groups,original_distinct_tags,groups_with_multiple_variants,excess_tag_variants
0,428275,539516.0,82681,111241.0


In [22]:

import os

sample_tables = {
    "product_records": "product_records_sample.csv",
    "product_countries": "product_countries_sample.csv",
    "product_categories": "product_categories_sample.csv",
    "product_brands": "product_brands_sample.csv",
}

for table_name, filename in sample_tables.items():
    output_path = f"/kaggle/working/{filename}"

    con.execute(f"""
        COPY (
            SELECT *
            FROM {table_name}
            ORDER BY record_id
            LIMIT 100
        )
        TO '{output_path}'
        (HEADER, DELIMITER ',')
    """)

    print(
        f"{filename}: "
        f"{os.path.getsize(output_path):,} bytes"
    )


product_records_sample.csv: 20,624 bytes
product_countries_sample.csv: 1,831 bytes
product_categories_sample.csv: 1,988 bytes
product_brands_sample.csv: 2,133 bytes


In [23]:

for table_name, filename in sample_tables.items():
    output_path = f"/kaggle/working/{filename}"

    result = con.execute("""
        SELECT COUNT(*) AS sample_rows
        FROM read_csv_auto(?)
    """, [output_path]).fetchdf()

    print(f"\n{table_name}")
    display(result)



product_records


,sample_rows
0,100



product_countries


,sample_rows
0,100



product_categories


,sample_rows
0,100



product_brands


,sample_rows
0,100


In [25]:
import duckdb
import os

local_path = "/kaggle/input/datasets/maulanakinditamami/open-food-facts-product-database-salin-judul/food.parquet"

assert os.path.exists(local_path), "File Parquet tidak ditemukan"

con = duckdb.connect()
print("DuckDB siap")
print("Dataset ditemukan:", os.path.exists(local_path))

DuckDB siap
Dataset ditemukan: True


In [26]:
con.execute("""
CREATE OR REPLACE TABLE product_records AS
SELECT
    ROW_NUMBER() OVER () AS record_id,
    code,
    brands,
    brands_tags,
    categories,
    categories_tags,
    countries_tags,
    completeness,
    created_t
FROM read_parquet(?)
""", [local_path])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [27]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT record_id) AS unique_record_ids,
    MIN(record_id) AS min_record_id,
    MAX(record_id) AS max_record_id
FROM product_records
""").fetchdf()

,total_rows,unique_record_ids,min_record_id,max_record_id
0,4828207,4828207,1,4828207


In [28]:
con.execute("""
CREATE OR REPLACE TABLE product_countries AS
SELECT DISTINCT
    p.record_id,
    t.country_tag
FROM product_records AS p
CROSS JOIN UNNEST(p.countries_tags) AS t(country_tag)
WHERE t.country_tag IS NOT NULL
  AND TRIM(t.country_tag) <> ''
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [29]:
con.execute("""
CREATE OR REPLACE TABLE product_categories AS
SELECT DISTINCT
    p.record_id,
    t.category_tag
FROM product_records AS p
CROSS JOIN UNNEST(p.categories_tags) AS t(category_tag)
WHERE t.category_tag IS NOT NULL
  AND TRIM(t.category_tag) <> ''
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [30]:
con.execute("""
CREATE OR REPLACE TABLE product_brands AS
SELECT DISTINCT
    p.record_id,
    t.brand_tag
FROM product_records AS p
CROSS JOIN UNNEST(p.brands_tags) AS t(brand_tag)
WHERE t.brand_tag IS NOT NULL
  AND TRIM(t.brand_tag) <> ''
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [31]:
con.execute("""
SELECT 'product_records' AS table_name, COUNT(*) AS row_count
FROM product_records
UNION ALL
SELECT 'product_countries', COUNT(*)
FROM product_countries
UNION ALL
SELECT 'product_categories', COUNT(*)
FROM product_categories
UNION ALL
SELECT 'product_brands', COUNT(*)
FROM product_brands
""").fetchdf()

,table_name,row_count
0,product_records,4828207
1,product_countries,5079894
2,product_categories,10480049
3,product_brands,3313249
